# Generate a review-first Databricks ingestion notebook

Offline path: use the included metadata-only fixture. Databricks path: use the active `spark` session. No credentials or source records are embedded.

## Prerequisites

Install `great-generator`. For SDK access outside a notebook, install `great-generator[databricks]` and use Databricks unified authentication. Metadata visibility and source object permissions are controlled by Unity Catalog.

In [ ]:
from pathlib import Path

from great_generator import StaticUnityCatalogClient

SOURCE_TABLE = "retail.sales.fact_orders"
FIXTURE = Path("examples/databricks/unity_catalog/mock_uc_metadata.json")
client = StaticUnityCatalogClient.from_path(FIXTURE)

# In a Databricks workspace, remove client=client and pass spark=spark instead.


## Generate synthetic data from metadata

In [ ]:
from tempfile import TemporaryDirectory

from great_generator import generate_uc_ingestion_code

with TemporaryDirectory() as output_dir:
    artifacts = generate_uc_ingestion_code(
        source_table=SOURCE_TABLE,
        include_related=True,
        target_catalog="synthetic_sandbox",
        target_schema="sales_demo",
        rows={"dim_customer": 25, "dim_product": 10, "fact_orders": 200},
        output_path=Path(output_dir) / "retail_pipeline.ipynb",
        seed=42,
        execute=False,
        client=client,
    )
    print(sorted(artifacts))


## Validate

In [ ]:
assert "notebook" in artifacts
# Generated writes are disabled and use error-if-exists semantics.


## Cleanup and limitations

These examples do not write tables. Remove only sandbox artifacts you explicitly create. Generated data is synthetic, not anonymized production data. Complex types and cyclic/self-referencing relationships require explicit modeling.